# NWM forecast demo: `streamflow-getter forecast`

Companion to [docs/FORECAST_CLI_PLAN.md](../docs/FORECAST_CLI_PLAN.md). It demonstrates every planned `forecast` command on one event: the **2026-04-19 Alabama** forecast (NWM v3.0), for two gaged reaches.

| Gage | Reach (`feature_id`) | Description |
|---|---|---|
| 02465000 | 18229923 | Black Warrior River at Northport/Tuscaloosa, just below Oliver Lock & Dam |
| 02428400 | 18239649 | Alabama River at Claiborne Lock & Dam near Monroeville |

Both reaches sit just below locks and dams, so their flow is regulated.

**Status: work in progress.** The `forecast` CLI does not exist yet. Each section shows the planned CLI command (commented out), then a Python stand-in that does the same job with what exists today: `nwm.gages_to_reaches` and `nwm_kerchunk.read_refs`. Stand-ins are labeled. Section 6 (cache) is not built.

**How to run:** needs the network and the `notebook` dependency group (`uv sync --group notebook`: matplotlib, ipykernel, h5netcdf, h5py). Open it with the project `.venv` as the kernel. Every cell has been run headlessly, but no outputs are saved in the file yet. It is not run in CI; committed outputs will come from a local run in Jupyter or VS Code.

In [ ]:
import re
import time

import numpy as np
import pandas as pd
import s3fs

from streamflow_getter import nwm
from streamflow_getter.nwm_kerchunk import REFS_ROOT, read_refs

DATE = "20260419"
CYCLE = pd.Timestamp("2026-04-19 12:00")  # forecast cycle (UTC)
GAGES = ["02465000", "02428400"]

s3 = s3fs.S3FileSystem(anon=True)

# Gage -> reach. These are retrospective v3.0 gage tags; v3.1 RouteLink tags differ
# for about 1,000 reaches (these two are probably unchanged). Takes a few seconds.
gage_map = nwm.gages_to_reaches(GAGES, nwm.open_chrtout())
REACHES = list(gage_map.values())
gage_map

## 1. What `forecast urls` resolves  *(stand-in)*

Resolves a selection to native S3/GCS/NOMADS URLs and the matching Kerchunk JSON path, without downloading anything. Needs `nwm_catalog.py` and `nwm_urls.py`.

The naming gotcha the catalog must encode: for medium range, the **directory** is `medium_range_memN`, but the **file** uses the base run name with the member on the product (`medium_range.channel_rt_N`).

In [ ]:
# CLI (planned):
# !streamflow-getter forecast urls --product short_range --cycle 2026-04-19T12 --domain conus

def ref_path(product, cycle, step, date=DATE):
    return f"{REFS_ROOT}/nwm.{date}/{product}/nwm.t{cycle:02d}z.{product}.channel_rt.{step}.conus.nc.json"


def mr_path(member, hour, date=DATE, cycle=12):
    return (f"{REFS_ROOT}/nwm.{date}/medium_range_mem{member}/"
            f"nwm.t{cycle:02d}z.medium_range.channel_rt_{member}.f{hour:03d}.conus.nc.json")


print(ref_path("short_range", 12, "f001"))
print(mr_path(2, 3))

## 2. `forecast list` for the day  *(stand-in)*

Lists which products and cycles exist for a date, and reports **partial days**: check that the last cycles (t23z for hourly products, t18z for long range) exist before treating a day as complete.

In [ ]:
# CLI (planned):
# !streamflow-getter forecast list --date 2026-04-19

day = f"{REFS_ROOT}/nwm.{DATE}"
all_dirs = [p.rsplit("/", 1)[1] for p in s3.ls(day)]
print(len(all_dirs), "product directories in the reference archive for", DATE)

rows = []
for d in ["analysis_assim", "short_range", "medium_range_mem1", "medium_range_mem2", "long_range_mem1"]:
    files = [p.rsplit("/", 1)[1] for p in s3.ls(f"{day}/{d}") if ".channel_rt" in p]
    cycles = sorted({re.match(r"nwm\.t(\d\d)z", f).group(1) for f in files})
    rows.append({"directory": d, "channel_rt_refs": len(files), "cycles": len(cycles), "last_cycle": cycles[-1]})
pd.DataFrame(rows)

## 3. `forecast fetch` with a gage, with and without t0

Fetches `streamflow` for the two gages' reaches from the t12z short-range cycle (f001 to f018) and, by default, a t0 row from the `analysis_assim` tm00 file of the same cycle.

- `lead_minutes` is computed **from the cycle**, not from the file's `reference_time`: `analysis_assim` files carry a `reference_time` 3 h before the cycle (model initialization). Lead is kept as integer minutes so 15-minute leads (Hawaii) match exactly; `lead_hours = lead_minutes / 60`.
- t0 rows keep `product = short_range` and have `source_product = analysis_assim` and `lead = 0`, so filtering on `product` keeps them.
- `member` is empty for products without ensemble members.
- `--no-t0` drops the `analysis_assim` rows.

In [ ]:
def to_long(df, product, cycle=CYCLE, member=pd.NA):
    """read_refs output -> the plan's long format."""
    lead_minutes = ((df["valid_time"] - cycle) // pd.Timedelta(minutes=1)).astype(int)
    out = df.assign(
        reference_time=cycle,
        lead_minutes=lead_minutes,
        lead_hours=lead_minutes / 60,
        member=member,
        product=product,
        source_product=df["ref_path"].str.split("/").str[-2],  # directory: analysis_assim / short_range
    )
    return out[["reference_time", "lead_minutes", "lead_hours", "valid_time", "member", "product",
                "source_product", "nwm_version", "feature_id", "streamflow"]]


# CLI (planned):
# !streamflow-getter forecast fetch --product short_range --cycle 2026-04-19T12 \
#     --gage 02465000 --gage 02428400 --out short_range.parquet

t0_path = ref_path("analysis_assim", CYCLE.hour, "tm00")
short_paths = [ref_path("short_range", CYCLE.hour, f"f{h:03d}") for h in range(1, 19)]

raw = read_refs([t0_path] + short_paths, REACHES)  # one call: feature_id is read once
with_t0 = to_long(raw, "short_range").sort_values(["feature_id", "lead_minutes"], ignore_index=True)
no_t0 = with_t0[with_t0["source_product"] != "analysis_assim"]

print("with t0:", len(with_t0), "rows; --no-t0:", len(no_t0), "rows; missing refs:", raw.attrs["missing"])
with_t0.head(5)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
for reach, grp in with_t0.groupby("feature_id"):
    line, = ax.plot(grp["lead_hours"], grp["streamflow"], marker="o", ms=3, label=str(reach))
    t0_row = grp[grp["source_product"] == "analysis_assim"]
    ax.plot(t0_row["lead_hours"], t0_row["streamflow"], marker="*", ms=14, ls="", color=line.get_color())
ax.set(xlabel="lead time (h)", ylabel="streamflow (m3/s)",
       title="short_range t12z 2026-04-19 (star = analysis_assim tm00, lead 0)")
ax.legend(title="feature_id")
plt.show()

### Kerchunk vs native  *(stand-in)*

`--source native` reads the NetCDF files directly. The two sources should return identical values for the same file. This needs `h5netcdf`.

In [ ]:
# CLI (planned):
# !streamflow-getter forecast fetch ... --source native --out native.parquet
import xarray as xr

native_key = "noaa-nwm-pds/nwm.20260419/short_range/nwm.t12z.short_range.channel_rt.f001.conus.nc"
with s3.open(native_key) as f:
    native = xr.open_dataset(f, engine="h5netcdf").streamflow.sel(feature_id=REACHES).values

kerchunk_f001 = with_t0[with_t0["lead_minutes"] == 60].set_index("feature_id").loc[REACHES, "streamflow"].values
print("native:   ", native)
print("kerchunk: ", kerchunk_f001)
assert np.allclose(native, kerchunk_f001, atol=0.01)

## 4. Medium-range ensemble spaghetti, with t0 marked  *(stand-in)*

Plots the six `medium_range` members for the first gage's reach (`mem1` is the control, in bold; `mem7` no longer exists), with the t0 value from `analysis_assim` marked. Members 2 to 6 end at f204; `mem1` runs to f240. Sampled every 3 hours to keep it to about 420 files (roughly a minute).

In [ ]:
# CLI (planned):
# !streamflow-getter forecast fetch --product medium_range --cycle 2026-04-19T12 \
#     --gage 02465000 --out medium_range.parquet

mr_paths = [mr_path(m, h) for m in range(1, 7) for h in range(3, 241, 3) if h <= (240 if m == 1 else 204)]
mr_raw = read_refs(mr_paths, REACHES[:1])
member = mr_raw["ref_path"].str.extract(r"channel_rt_(\d)")[0].astype(int)
mr = to_long(mr_raw, "medium_range").assign(member=member.values)
mr.groupby("member").size()

In [ ]:
import matplotlib.pyplot as plt  # needs matplotlib

fig, ax = plt.subplots()
for m, grp in mr.groupby("member"):
    grp = grp.sort_values("valid_time")
    ax.plot(grp["valid_time"], grp["streamflow"], lw=2.5 if m == 1 else 1, color="k" if m == 1 else None,
            label=f"mem{m}")
t0_row = with_t0[(with_t0["feature_id"] == REACHES[0]) & (with_t0["source_product"] == "analysis_assim")]
ax.plot(t0_row["valid_time"], t0_row["streamflow"], marker="*", ms=14, ls="", color="r", label="t0 (analysis_assim)")
ax.set(ylabel="streamflow (m3/s)", title=f"medium_range t12z 2026-04-19, feature_id {REACHES[0]}")
ax.legend()
fig.autofmt_xdate()
plt.show()

## 5. `forecast bench`: Kerchunk vs native  *(stand-in)*

Times Kerchunk (only the chunks that hold the requested reaches) against native (whole NetCDF files) over the 18 short-range files. The cell below is a mini-benchmark on your machine: both sides use 8 threads, and "native" reads byte ranges through `h5netcdf` (not whole files). It needs `h5netcdf`.

For reference, medians from the full benchmark on the upstream `bench-kerchunk` branch (PR #4, still open, so numbers may change): 6 Alabama reaches, 18 short-range files, Apple M4 Max on Wi-Fi. Script: `benchmarks/kerchunk_vs_native.py`; write-up: `docs/benchmarks/kerchunk-vs-native.md`. `read_refs` uses the `kerchunk_direct` approach.

| Date (version) | native_full | native_range | kerchunk_gcs | kerchunk_direct |
|---|---|---|---|---|
| 2026-04-19 (v3.0) | 17.5 s / 257 MB | 23.7 s / 67 MB | 4.4 s / 48 MB | 5.4 s / 37 MB |
| 2026-10-27 (v3.1) | 11.6 s / 223 MB | 8.5 s / 47 MB | 3.1 s / 31 MB | 3.4 s / 20 MB |

Medium range `mem1`, 240 files (v3.0): `native_range` 172.5 s vs `kerchunk_direct` 31.5 s.

In [ ]:
# CLI (planned):
# !streamflow-getter forecast bench --product short_range --cycle 2026-04-19T12 --gage 02465000

start = time.perf_counter()
read_refs(short_paths, REACHES)
kerchunk_s = time.perf_counter() - start

import xarray as xr  # needs h5netcdf
from concurrent.futures import ThreadPoolExecutor


def read_native(h):
    key = f"noaa-nwm-pds/nwm.20260419/short_range/nwm.t12z.short_range.channel_rt.f{h:03d}.conus.nc"
    with s3.open(key) as f:  # byte-range reads, like the advisor's native_range method
        return xr.open_dataset(f, engine="h5netcdf").streamflow.sel(feature_id=REACHES).values


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:  # same 8 threads as read_refs
    list(pool.map(read_native, range(1, 19)))
native_s = time.perf_counter() - start

pd.DataFrame({"source": ["kerchunk", "native (byte ranges)"], "seconds_for_18_files": [round(kerchunk_s, 1), round(native_s, 1)]})

## 6. Cache management: `forecast refs build/clear`  *(not built)*

Builds, updates and clears a local cache of Kerchunk references (default `~/.cache/streamflow_getter`, configurable). Will show cache size before and after.

In [ ]:
# CLI (planned):
# !streamflow-getter forecast refs build --product short_range --cycle 2026-04-19T12
# !streamflow-getter forecast refs clear